# Week 26, Day 2: Review II: eigenvalues, the SVD and calculus

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w026/d2.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Problem 1: which pattern does the network remember?

A network of 50 rate neurons obeys τ dx/dt = −x + Wx with a symmetric W (τ = 10 ms). Find the eigenvector of W with the largest eigenvalue, `v_slow`, and its predicted time constant `tau_slow` = τ / (1 − λmax). Then simulate 200 ms (Euler, dt = 0.1 ms) from x = v_slow and from x = the eigenvector with the smallest eigenvalue, and store the fraction of the starting norm left in `keep_slow` and `keep_fast`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
r1 = np.random.default_rng(10)
M = r1.normal(size=(50, 50)) / np.sqrt(50)
W = 0.3 * (M + M.T)                            # symmetric connectivity
tau = 10.0
lam, V = np.linalg.eigh(W)                       # eigenvalues in ascending order
v_slow = ...
tau_slow = ...

def simulate(x0, T=200.0, dt=0.1):
    x = x0.copy()
    for _ in range(int(T / dt)):
        x = x + dt / tau * (-x + W @ x)
    return np.linalg.norm(x) / np.linalg.norm(x0)

keep_slow, keep_fast = simulate(v_slow), simulate(V[:, 0])
print(f"largest eigenvalue {lam[-1]:.3f}: predicted time constant {tau_slow:.1f} ms;  left after 200 ms: slow {keep_slow:.3f}, fast {keep_fast:.2e}")

In [ ]:
import numpy as np
def test_slow():
    "v_slow is the unit eigenvector of W with the largest eigenvalue, and tau_slow its time constant"
    lam, V = np.linalg.eigh(W)
    assert abs(abs(v_slow @ V[:, -1]) - 1) < 1e-9 and abs(tau_slow - tau / (1 - lam[-1])) < 1e-9

def test_sim():
    "after 200 ms the simulated state started along v_slow keeps over 100 times more of its size than one started along the fastest eigenvector"
    assert keep_slow > 100 * keep_fast

test_slow()
test_sim()
print("All checks passed")

### Exercise: Problem 2: the true rank under noise

A 100 × 40 matrix L of rank 3 (say, 100 time points of 40 neurons driven by 3 latent signals) is observed with noise as Y. For k = 1 to 10, compute the relative error ‖L_k − L‖/‖L‖ of the rank-k truncated SVD of Y and store it in `errs`. Look at the singular values for a gap.

In [ ]:
r2 = np.random.default_rng(7)
L = r2.normal(size=(100, 3)) @ (r2.normal(size=(3, 40)) * np.array([[5.0], [3.0], [2.0]]))
Y = L + 1.0 * r2.normal(size=L.shape)
U, S, Vt = np.linalg.svd(Y, full_matrices=False)
errs = []
for k in range(1, 11):
    Lk = ...
    errs.append(np.linalg.norm(Lk - L) / np.linalg.norm(L))
print("singular values:", S[:8].round(1))
print("error against the truth by rank:", np.round(errs, 3))

In [ ]:
import numpy as np
def test_errors():
    "errs[k-1] is the error of the rank-k truncated SVD of Y against the clean signal L"
    U, S, Vt = np.linalg.svd(Y, full_matrices=False)
    for k in (1, 3, 6):
        Lk = U[:, :k] * S[:k] @ Vt[:k]
        assert abs(errs[k - 1] - np.linalg.norm(Lk - L) / np.linalg.norm(L)) < 1e-9

def test_best():
    "the error against the truth is smallest at rank 3, the true rank"
    assert int(np.argmin(errs)) + 1 == 3

test_errors()
test_best()
print("All checks passed")

### Exercise: Problem 3: a gradient by hand, checked

For logistic regression, the loss is L(w) = −Σ [y log p + (1 − y) log(1 − p)] with p = σ(Xw). Use the chain rule to show that its gradient is Xᵀ(p − y) and its Hessian is Xᵀ diag(p(1 − p)) X, implement `grad` and `hess`, and check them.

In [ ]:
r3 = np.random.default_rng(8)
X = r3.normal(size=(60, 5)); y = (r3.random(60) < 0.5).astype(float)
sig = lambda z: 1 / (1 + np.exp(-z))

def loss(w):
    p = sig(X @ w)
    return -np.sum(y * np.log(p) + (1 - y) * np.log(1 - p))

def grad(w):
    return ...

def hess(w):
    p = sig(X @ w)
    return ...

w0 = np.zeros(5)
print("gradient at 0:", grad(w0).round(3), " smallest Hessian eigenvalue:", np.linalg.eigvalsh(hess(w0)).min().round(3))

In [ ]:
import numpy as np
def test_grad():
    "grad matches central finite differences to 1e-6"
    w = np.random.default_rng(1).normal(size=5)
    num = np.array([(loss(w + 1e-6 * e) - loss(w - 1e-6 * e)) / 2e-6 for e in np.eye(5)])
    assert np.allclose(grad(w), num, atol=1e-6)

def test_hess():
    "hess is symmetric with non-negative eigenvalues (the loss is convex)"
    H = hess(np.random.default_rng(2).normal(size=5))
    assert np.allclose(H, H.T) and np.linalg.eigvalsh(H).min() > -1e-10

test_grad()
test_hess()
print("All checks passed")